In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Bronze — Sales Orders
# MAGIC **Fonte:** `/databricks-datasets/retail-org/sales_orders/` (JSON)
# MAGIC **Destino:** `retail_dev.bronze.bronze_sales_orders` (Delta)
# MAGIC
# MAGIC ### Estrutura do notebook
# MAGIC 1. Setup
# MAGIC 2. Schema e Volume
# MAGIC 3. Amostra
# MAGIC 4. Qualidade — nulos e unicidade
# MAGIC 5. Estrutura de `ordered_products` — array de itens
# MAGIC 6. Ingestão → Delta
# MAGIC 7. Verificação

# COMMAND ----------

# MAGIC %md
# MAGIC ---
# MAGIC ## 1. Setup

# COMMAND ----------

from pyspark.sql import functions as F

SOURCE_PATH = "/databricks-datasets/retail-org/sales_orders/" ## Repositório do databricks
TARGET_TABLE = "retail_dev.bronze.bronze_sales_orders" ## Tabela de destino na bronze


In [0]:

# COMMAND ----------

# MAGIC %md
# MAGIC ---
# MAGIC ## 2. Schema e Volume

# COMMAND ----------

df = spark.read.json(SOURCE_PATH)

print("=== SCHEMA ===")
df.printSchema()

print(f"\n=== VOLUME ===")
print(f"Total de linhas:   {df.count()}")
print(f"Total de colunas:  {len(df.columns)}")
print(f"Colunas:           {df.columns}")


## clicked_items
## 

In [0]:
# COMMAND ----------

# MAGIC %md
# MAGIC ---
# MAGIC ## 3. Amostra

# COMMAND ----------

display(df.limit(5))

In [0]:
display(
    df.select(
        F.count("*").alias("total_pedidos"),
        F.countDistinct("order_number").alias("pedidos_unicos"),
        F.countDistinct("customer_id").alias("clientes_unicos"),
        F.sum(F.when(F.col("order_number").isNull(), 1).otherwise(0)).alias("order_number_nulos"),
        F.sum(F.when(F.col("customer_id").isNull(), 1).otherwise(0)).alias("customer_id_nulos"),
        F.sum(F.when(F.col("order_datetime").isNull(), 1).otherwise(0)).alias("order_datetime_nulos"),
        F.sum(F.when(F.col("ordered_products").isNull(), 1).otherwise(0)).alias("ordered_products_nulos"),
        F.sum(F.when(F.col("number_of_line_items").isNull(), 1).otherwise(0)).alias("number_of_line_items_nulos"),
    )
)

# MAGIC ### Pedidos duplicados?

display(
    df.groupBy("order_number")
    .agg(F.count("*").alias("ocorrencias"))
    .filter(F.col("ocorrencias") > 1)
    .orderBy(F.col("ocorrencias").desc())
)

# MAGIC ### Tipo de `order_datetime` — confirmar que é string

# COMMAND ----------

display(
    df.select(
        F.col("order_datetime"),
        F.col("number_of_line_items"),
    ).limit(10)
)


In [0]:

# MAGIC ## 5. Estrutura de `ordered_products`
# MAGIC Array de structs — cada elemento é um item do pedido.
# MAGIC ### Schema do array
df.select("ordered_products").printSchema()


# MAGIC ### Explode — um item por linha

display(
    df
    .withColumn("item", F.explode("ordered_products"))
    .select("order_number", "customer_id", "item.*")
    .limit(15)
)

# COMMAND ----------

# MAGIC %md
# MAGIC ### Campos disponíveis dentro de cada item

# COMMAND ----------

df_items = (
    df
    .withColumn("item", F.explode("ordered_products"))
    .select("order_number", "item.*")
)

print("Colunas após explode:")
for col in df_items.columns:
    print(f"  {col}")

# COMMAND ----------

# MAGIC %md
# MAGIC ### Distribuição de itens por pedido

# COMMAND ----------

display(
    df
    .withColumn("qtd_itens", F.size("ordered_products"))
    .groupBy("qtd_itens")
    .agg(F.count("*").alias("total_pedidos"))
    .orderBy("qtd_itens")
)

# COMMAND ----------

# MAGIC %md
# MAGIC ### Período de dados — range de datas

# COMMAND ----------

display(
    df.select(
        F.min("order_datetime").alias("data_mais_antiga"),
        F.max("order_datetime").alias("data_mais_recente"),
    )
)

# COMMAND ----------


In [0]:

# MAGIC ## 6. Ingestão → Delta
# MAGIC Escrita na camada bronze — estrutura raw preservada, zero transformações.
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print(f"Tabela escrita: {TARGET_TABLE}")


In [0]:
df_check = spark.table(TARGET_TABLE)

print(f"Linhas na tabela: {df_check.count()}")
print(f"Colunas:          {df_check.columns}")

display(df_check.limit(5))
